# Lab 4.3: Tool Distribution & tool_choice Config

**What you'll build:** A multi-agent tool distribution scheme that keeps each agent at 4-5 tools -- and learn why 18 tools on one agent degrades selection.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- 18 tools on one agent causes misselection | 5 min |
| 2 | The Right Way -- distributed agents with scoped tools | 5 min |
| 3 | Your Turn -- design a tool distribution for a new multi-agent system | 10 min |
| 4 | tool_choice -- configure forced vs auto selection for pipeline steps | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a customer operations system with tools for support, billing, inventory, and shipping. We'll first give ALL tools to a single agent, then distribute them across focused agents.

The test: 12 queries across 4 domains. We measure tool selection accuracy.

In [ ]:
def make_tool(name, desc):
    """Helper to create a minimal tool definition."""
    return {
        "name": name,
        "description": desc,
        "input_schema": {
            "type": "object",
            "properties": {"input": {"type": "string"}},
            "required": ["input"]
        }
    }

# All 18 tools -- simulating a monolith agent
ALL_TOOLS = [
    # Support (5)
    make_tool("lookup_ticket", "Looks up a support ticket."),
    make_tool("create_ticket", "Creates a support ticket."),
    make_tool("escalate_ticket", "Escalates a ticket to a manager."),
    make_tool("get_faq", "Gets FAQ articles."),
    make_tool("send_support_email", "Sends a support email."),
    # Billing (5)
    make_tool("get_invoice", "Gets an invoice."),
    make_tool("process_payment", "Processes a payment."),
    make_tool("issue_refund", "Issues a refund."),
    make_tool("update_billing_info", "Updates billing information."),
    make_tool("get_payment_history", "Gets payment history."),
    # Inventory (4)
    make_tool("check_stock", "Checks stock levels."),
    make_tool("reserve_item", "Reserves an item."),
    make_tool("get_product_info", "Gets product information."),
    make_tool("search_catalog", "Searches the product catalog."),
    # Shipping (4)
    make_tool("track_package", "Tracks a package."),
    make_tool("get_shipping_rates", "Gets shipping rates."),
    make_tool("create_shipment", "Creates a shipment."),
    make_tool("update_address", "Updates shipping address."),
]

TEST_QUERIES = [
    {"query": "I need to check on my support ticket #4567", "expected": "lookup_ticket", "domain": "support"},
    {"query": "Can you look up my latest invoice?", "expected": "get_invoice", "domain": "billing"},
    {"query": "Is the blue widget in stock?", "expected": "check_stock", "domain": "inventory"},
    {"query": "Where is my package right now?", "expected": "track_package", "domain": "shipping"},
    {"query": "I want a refund for my last order", "expected": "issue_refund", "domain": "billing"},
    {"query": "Create a new support ticket about my broken item", "expected": "create_ticket", "domain": "support"},
    {"query": "What are the shipping options for a 5kg package?", "expected": "get_shipping_rates", "domain": "shipping"},
    {"query": "Show me details about product SKU-9876", "expected": "get_product_info", "domain": "inventory"},
    {"query": "I need to update my credit card on file", "expected": "update_billing_info", "domain": "billing"},
    {"query": "This issue needs to go to a manager", "expected": "escalate_ticket", "domain": "support"},
    {"query": "Reserve 3 units of product ABC-123 for me", "expected": "reserve_item", "domain": "inventory"},
    {"query": "I need to change my delivery address", "expected": "update_address", "domain": "shipping"},
]

print(f"Total tools: {len(ALL_TOOLS)}")
print(f"Test queries: {len(TEST_QUERIES)}")

---

## Phase 1: The Wrong Approach

Give all 18 tools to a single agent.

In [ ]:
def test_selection(tools, queries, label):
    """Test tool selection accuracy."""
    results = []
    for q in queries:
        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            tools=tools,
            tool_choice={"type": "any"},
            messages=[{"role": "user", "content": q["query"]}],
        )
        selected = None
        for block in response.content:
            if block.type == "tool_use":
                selected = block.name
                break
        results.append({
            "query": q["query"], "expected": q["expected"],
            "selected": selected, "correct": selected == q["expected"],
            "domain": q.get("domain", "")
        })
    
    correct = sum(1 for r in results if r["correct"])
    print(f"\n=== {label} ===")
    print(f"Accuracy: {correct}/{len(results)} ({correct/len(results):.0%})")
    for r in results:
        tag = "OK" if r["correct"] else "XX"
        print(f"  [{tag}] [{r['domain']}] \"{r['query'][:45]}\"")
        if not r["correct"]:
            print(f"         Expected: {r['expected']} | Got: {r['selected']}")
    return results

monolith_results = test_selection(ALL_TOOLS, TEST_QUERIES, "MONOLITH: 18 TOOLS, 1 AGENT")

---

## Phase 2: The Right Approach

Distribute tools across 4 domain-specific agents. Each gets 4-5 tools. A simple router picks the right agent.

In [ ]:
# Domain-specific tool sets (4-5 each)
AGENT_TOOLS = {
    "support": [
        make_tool("lookup_ticket", "Looks up a support ticket by ticket number. Use for checking ticket status."),
        make_tool("create_ticket", "Creates a new support ticket. Use when user reports an issue or problem."),
        make_tool("escalate_ticket", "Escalates a ticket to management. Use when user requests a manager."),
        make_tool("get_faq", "Searches FAQ articles. Use for common questions about policies or procedures."),
        make_tool("send_support_email", "Sends a follow-up email to the customer about their support case."),
    ],
    "billing": [
        make_tool("get_invoice", "Retrieves invoice details by invoice number or date range."),
        make_tool("process_payment", "Processes a payment for an outstanding balance."),
        make_tool("issue_refund", "Issues a refund for a previous charge. Use when user requests money back."),
        make_tool("update_billing_info", "Updates payment method or billing address on file."),
        make_tool("get_payment_history", "Retrieves list of past payments and transactions."),
    ],
    "inventory": [
        make_tool("check_stock", "Checks current stock level for a product. Use for availability questions."),
        make_tool("reserve_item", "Reserves inventory units for a customer. Use when user wants to hold items."),
        make_tool("get_product_info", "Gets detailed product information including specs, price, and description."),
        make_tool("search_catalog", "Searches the product catalog by keyword, category, or attribute."),
    ],
    "shipping": [
        make_tool("track_package", "Tracks package location and delivery status by tracking number."),
        make_tool("get_shipping_rates", "Calculates shipping options and costs for a given weight and destination."),
        make_tool("create_shipment", "Creates a new shipment label and schedules pickup."),
        make_tool("update_address", "Updates the shipping/delivery address for an order."),
    ],
}

# Simple router: use Claude to pick the domain, then call the domain agent
ROUTER_TOOLS = [
    make_tool("route_to_support", "Route to support agent. Use when user has a ticket, issue, or needs help."),
    make_tool("route_to_billing", "Route to billing agent. Use for payments, invoices, refunds, billing info."),
    make_tool("route_to_inventory", "Route to inventory agent. Use for stock, products, catalog, reservations."),
    make_tool("route_to_shipping", "Route to shipping agent. Use for tracking, delivery, shipping rates, address changes."),
]

DOMAIN_MAP = {
    "route_to_support": "support",
    "route_to_billing": "billing",
    "route_to_inventory": "inventory",
    "route_to_shipping": "shipping",
}


def distributed_selection(query, expected):
    """Two-step: router -> domain agent."""
    # Step 1: Route
    route_resp = client.messages.create(
        model=MODEL, max_tokens=256,
        tools=ROUTER_TOOLS, tool_choice={"type": "any"},
        messages=[{"role": "user", "content": query}],
    )
    route_tool = None
    for block in route_resp.content:
        if block.type == "tool_use":
            route_tool = block.name
            break
    
    domain = DOMAIN_MAP.get(route_tool, "support")
    domain_tools = AGENT_TOOLS[domain]
    
    # Step 2: Domain agent selects specific tool
    agent_resp = client.messages.create(
        model=MODEL, max_tokens=256,
        tools=domain_tools, tool_choice={"type": "any"},
        messages=[{"role": "user", "content": query}],
    )
    selected = None
    for block in agent_resp.content:
        if block.type == "tool_use":
            selected = block.name
            break
    
    return {"domain": domain, "selected": selected, "correct": selected == expected}


print("Testing distributed agent architecture...\n")
dist_results = []
for q in TEST_QUERIES:
    r = distributed_selection(q["query"], q["expected"])
    r["query"] = q["query"]
    r["expected"] = q["expected"]
    dist_results.append(r)
    tag = "OK" if r["correct"] else "XX"
    print(f"  [{tag}] [{r['domain']}] \"{q['query'][:45]}\"")
    if not r["correct"]:
        print(f"         Expected: {q['expected']} | Got: {r['selected']}")

dist_correct = sum(1 for r in dist_results if r["correct"])
mono_correct = sum(1 for r in monolith_results if r["correct"])
total = len(TEST_QUERIES)

print(f"\n=== COMPARISON ===")
print(f"Monolith (18 tools): {mono_correct}/{total} ({mono_correct/total:.0%})")
print(f"Distributed (4-5/agent): {dist_correct}/{total} ({dist_correct/total:.0%})")

---

## Phase 3: Your Turn

Design a tool distribution for a content creation system with 15 tools across 3 roles.

In [ ]:
# All 15 content creation tools
CONTENT_TOOLS = [
    # Research
    "search_sources", "fact_check", "get_statistics", "summarize_article", "find_quotes",
    # Writing
    "draft_article", "rewrite_section", "generate_headline", "add_transition", "check_grammar",
    # Publishing
    "format_html", "optimize_seo", "schedule_publish", "create_social_post", "generate_thumbnail",
]

CONTENT_QUERIES = [
    {"query": "Find 3 reliable sources about renewable energy trends", "expected": "search_sources"},
    {"query": "Write a first draft about solar panel efficiency", "expected": "draft_article"},
    {"query": "Format this article for our WordPress site", "expected": "format_html"},
    {"query": "Is this claim about CO2 levels accurate?", "expected": "fact_check"},
    {"query": "Rewrite the introduction to be more engaging", "expected": "rewrite_section"},
    {"query": "Schedule this post for Monday at 9am", "expected": "schedule_publish"},
    {"query": "Generate 5 headline options for this article", "expected": "generate_headline"},
    {"query": "Optimize the meta description for search engines", "expected": "optimize_seo"},
    {"query": "Create a tweet thread promoting this article", "expected": "create_social_post"},
]

print("Challenge: Distribute 15 content tools across 3 focused agents.")
print("Each agent should have 4-5 tools. You may add 1 cross-role tool if needed.")
print(f"\nTools to distribute: {CONTENT_TOOLS}")

In [ ]:
# =============================================================
# TODO: Design your tool distribution
# =============================================================
#
# Requirements:
# - 3 agents: research, writing, publishing
# - Each agent gets 4-5 tools (max 5 including any cross-role tools)
# - You may add 1 cross-role tool to agents that need it
#
# Think about:
# - Which tools naturally group by domain?
# - Does any tool need to be available to multiple agents?
# - How do you route queries to the right agent?

YOUR_DISTRIBUTION = {
    "research": {
        "tools": [],   # TODO: list tool names for this agent
        "description": ""  # TODO: describe what this agent handles
    },
    "writing": {
        "tools": [],
        "description": ""
    },
    "publishing": {
        "tools": [],
        "description": ""
    },
}

# Validate your distribution
all_distributed = set()
issues = []
for agent, config in YOUR_DISTRIBUTION.items():
    tools = config["tools"]
    if len(tools) == 0:
        issues.append(f"  {agent}: No tools assigned (TODO not completed)")
    elif len(tools) > 5:
        issues.append(f"  {agent}: {len(tools)} tools (max 5)")
    all_distributed.update(tools)

missing = set(CONTENT_TOOLS) - all_distributed
if missing:
    issues.append(f"  Unassigned tools: {missing}")

if issues:
    print("Issues with your distribution:")
    for i in issues:
        print(i)
else:
    print("Distribution looks valid!")
    for agent, config in YOUR_DISTRIBUTION.items():
        print(f"  {agent}: {len(config['tools'])} tools -- {config['tools']}")

---

## Phase 4: tool_choice Configuration

Experiment with the three tool_choice modes to understand when to use each.

In [ ]:
# Demonstrate tool_choice modes
EXTRACT_TOOLS = [
    {
        "name": "extract_metadata",
        "description": "Extracts document metadata: title, author, date, category.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "author": {"type": "string"},
                "date": {"type": "string"},
                "category": {"type": "string"}
            },
            "required": ["title"]
        }
    },
    {
        "name": "extract_entities",
        "description": "Extracts named entities: people, organizations, locations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "people": {"type": "array", "items": {"type": "string"}},
                "organizations": {"type": "array", "items": {"type": "string"}},
                "locations": {"type": "array", "items": {"type": "string"}}
            }
        }
    },
]

doc = """Climate Change Report 2024 by Dr. Sarah Chen, MIT Climate Lab.
Published March 15, 2024. Category: Environmental Science.
Key findings from research conducted in Boston, London, and Tokyo."""

# Mode 1: auto (model decides IF it calls a tool)
print("=== tool_choice: auto ===")
resp_auto = client.messages.create(
    model=MODEL, max_tokens=512,
    tools=EXTRACT_TOOLS,
    tool_choice={"type": "auto"},
    messages=[{"role": "user", "content": f"Here's a document: {doc}"}],
)
print(f"Stop reason: {resp_auto.stop_reason}")
for block in resp_auto.content:
    if block.type == "tool_use":
        print(f"Called: {block.name}")
    elif block.type == "text":
        print(f"Text: {block.text[:100]}")

# Mode 2: any (model MUST call a tool, picks which)
print("\n=== tool_choice: any ===")
resp_any = client.messages.create(
    model=MODEL, max_tokens=512,
    tools=EXTRACT_TOOLS,
    tool_choice={"type": "any"},
    messages=[{"role": "user", "content": f"Extract data from: {doc}"}],
)
print(f"Stop reason: {resp_any.stop_reason}")
for block in resp_any.content:
    if block.type == "tool_use":
        print(f"Called: {block.name} -> {json.dumps(block.input, indent=2)[:200]}")

# Mode 3: forced (model MUST call extract_metadata specifically)
print("\n=== tool_choice: forced (extract_metadata) ===")
resp_forced = client.messages.create(
    model=MODEL, max_tokens=512,
    tools=EXTRACT_TOOLS,
    tool_choice={"type": "tool", "name": "extract_metadata"},
    messages=[{"role": "user", "content": f"Process this document: {doc}"}],
)
print(f"Stop reason: {resp_forced.stop_reason}")
for block in resp_forced.content:
    if block.type == "tool_use":
        print(f"Called: {block.name} -> {json.dumps(block.input, indent=2)}")

### Key Takeaways

1. **18 tools on one agent degrades selection.** The 4-5 tool limit is about decision complexity. Scope each agent to its role.
2. **Use a coordinator/router pattern.** A lightweight router (4 routing tools) picks the domain, then the domain agent (4-5 tools) picks the specific tool.
3. **Cross-role tools avoid routing overhead.** If one tool is needed by multiple agents (e.g., verify_fact), add it to each rather than routing back through the coordinator.
4. **tool_choice modes:** `auto` for conversational agents, `any` for pipeline steps that must call a tool, forced `{name: X}` when the exact tool is predetermined.
5. **Forced tool_choice guarantees pipeline ordering.** Use it when Step 1 must always run before Step 2.